In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Locate project folder
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Read cleaned dataset
csv_path = project_root / "data" / "hotel_bookings_clean.csv"
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Row IDs unique:", df["row_id"].is_unique)

In [ ]:
# Define target variable
y = df["is_canceled"]

# Define input features
X = df.drop(columns=["is_canceled", "row_id"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nCancellation distribution:")
print(y.value_counts(normalize=True).round(4))

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print("Training target:", y_train.shape)
print("Test target:", y_test.shape)

print("\nTraining cancellation rate:", round(y_train.mean() * 100, 2), "%")
print("Test cancellation rate:", round(y_test.mean() * 100, 2), "%")

In [ ]:
# Create folder for split files
split_dir = project_root / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

# Save original row IDs for training set
train_ids = df.loc[X_train.index, ["row_id"]]
train_ids.to_csv(
    split_dir / "train_ids.csv",
    index=False
)

# Save original row IDs for test set
test_ids = df.loc[X_test.index, ["row_id"]]
test_ids.to_csv(
    split_dir / "test_ids.csv",
    index=False
)

print("Training IDs saved:", len(train_ids))
print("Test IDs saved:", len(test_ids))
print("Files saved to:", split_dir)

In [ ]:
# Check whether train and test sets overlap
overlap = set(train_ids["row_id"]) & set(test_ids["row_id"])

# Check whether all records are included
total_split = len(train_ids) + len(test_ids)

print("Overlapping records:", len(overlap))
print("Total split records:", total_split)
print("Original records:", len(df))

print("All records accounted for:", total_split == len(df))

In [ ]:
# Reload saved split IDs
train_check = pd.read_csv(split_dir / "train_ids.csv")
test_check = pd.read_csv(split_dir / "test_ids.csv")

print("Train file shape:", train_check.shape)
print("Test file shape:", test_check.shape)

print("Train IDs unique:", train_check["row_id"].is_unique)
print("Test IDs unique:", test_check["row_id"].is_unique)

In [ ]:
# Create a table containing row IDs and dataset assignments
random_split = pd.DataFrame({
    "row_id": df["row_id"],
    "split": "train"
})

# Mark test records
random_split.loc[X_test.index, "split"] = "test"

# Check the split
print(random_split["split"].value_counts())
print("Unique row IDs:", random_split["row_id"].is_unique)

In [ ]:
# Create a folder for split files
splits_dir = project_root / "data" / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

# Save random split assignments
random_split_path = splits_dir / "random_split.csv"
random_split.to_csv(random_split_path, index=False)

print("Saved to:", random_split_path)
print("File exists:", random_split_path.exists())

In [ ]:
# Build the scheduled arrival date specified in the Proposal
scheduled_arrival_date = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["arrival_date_month"] + "-" +
    df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d",
    errors="raise"
)

assert scheduled_arrival_date.notna().all()

print("Scheduled arrival date coverage:")
print("Earliest:", scheduled_arrival_date.min())
print("Latest:", scheduled_arrival_date.max())
print("Missing dates:", scheduled_arrival_date.isna().sum())


In [ ]:
# Compare pre-specified month-end candidates using data characteristics only.
# No model is fitted and no performance metric is used to choose the cutoff.
candidate_cutoffs = pd.to_datetime([
    "2017-03-31",
    "2017-04-30",
    "2017-05-31"
])

dated_df = df.assign(scheduled_arrival_date=scheduled_arrival_date)
comparison_rows = []

for candidate in candidate_cutoffs:
    for split_name, mask in {
        "train": scheduled_arrival_date <= candidate,
        "test": scheduled_arrival_date > candidate
    }.items():
        part = dated_df.loc[mask]
        hotel_share = part["hotel"].value_counts(normalize=True)
        comparison_rows.append({
            "cutoff": candidate.date().isoformat(),
            "split": split_name,
            "rows": len(part),
            "share_pct": round(len(part) / len(df) * 100, 2),
            "cancellation_rate_pct": round(part["is_canceled"].mean() * 100, 2),
            "lead_time_mean": round(part["lead_time"].mean(), 2),
            "lead_time_median": round(part["lead_time"].median(), 2),
            "city_hotel_pct": round(hotel_share.get("City Hotel", 0) * 100, 2),
            "resort_hotel_pct": round(hotel_share.get("Resort Hotel", 0) * 100, 2),
            "date_min": part["scheduled_arrival_date"].min().date().isoformat(),
            "date_max": part["scheduled_arrival_date"].max().date().isoformat()
        })

candidate_comparison = pd.DataFrame(comparison_rows)
candidate_comparison


In [ ]:
# Selection rule fixed before downstream modeling:
# 1. use a calendar month-end;
# 2. keep at least four complete future calendar months and at least 15% of rows for testing;
# 3. retain both hotel types in both partitions;
# 4. among eligible candidates, choose the test share closest to the intended 20% holdout.
# Model performance is deliberately not part of this rule.

candidate_summary = []
for candidate in candidate_cutoffs:
    test_mask = scheduled_arrival_date > candidate
    test_periods = scheduled_arrival_date.loc[test_mask].dt.to_period("M").nunique()
    test_share = test_mask.mean()
    both_hotels = all(
        df.loc[mask, "hotel"].nunique() == 2
        for mask in (scheduled_arrival_date <= candidate, test_mask)
    )
    candidate_summary.append({
        "cutoff": candidate,
        "test_share": test_share,
        "complete_future_months": test_periods,
        "both_hotels_in_both_splits": both_hotels,
        "eligible": test_share >= 0.15 and test_periods >= 4 and both_hotels
    })

selection_table = pd.DataFrame(candidate_summary)
eligible = selection_table.loc[selection_table["eligible"]].copy()
eligible["distance_from_20pct"] = (eligible["test_share"] - 0.20).abs()
cutoff_date = eligible.sort_values("distance_from_20pct").iloc[0]["cutoff"]

assert cutoff_date == pd.Timestamp("2017-04-30")
print(selection_table)
print("\nSelected scheduled-arrival cutoff:", cutoff_date.date())
print("Reason: closest eligible test share to 20%, with four complete future months.")


In [ ]:
# Assign the chronological robustness split by scheduled arrival date.
# Training includes arrivals through the selected cutoff; testing begins the next day.
chronological_split = pd.DataFrame({
    "row_id": df["row_id"],
    "scheduled_arrival_date": scheduled_arrival_date,
    "split": "test"
})

chronological_split.loc[
    chronological_split["scheduled_arrival_date"] <= cutoff_date,
    "split"
] = "train"

print(chronological_split["split"].value_counts())
print("\nSplit percentages:")
print(chronological_split["split"].value_counts(normalize=True).mul(100).round(2))


In [ ]:
# Validate completeness, uniqueness, and the strict temporal boundary.
train_dates = chronological_split.loc[
    chronological_split["split"] == "train", "scheduled_arrival_date"
]
test_dates = chronological_split.loc[
    chronological_split["split"] == "test", "scheduled_arrival_date"
]

assert chronological_split["row_id"].is_unique
assert set(chronological_split["row_id"]) == set(df["row_id"])
assert set(chronological_split["split"]) == {"train", "test"}
assert train_dates.max() == pd.Timestamp("2017-04-30")
assert test_dates.min() == pd.Timestamp("2017-05-01")
assert train_dates.max() < test_dates.min()
assert (chronological_split["split"] == "train").sum() == 96428
assert (chronological_split["split"] == "test").sum() == 22136

print("Latest training arrival:", train_dates.max())
print("Earliest test arrival:", test_dates.min())
print("Chronological order correct: True")
print("All row IDs included exactly once: True")


In [ ]:
# Report the selected split's descriptive checks.
selected_comparison = candidate_comparison.loc[
    candidate_comparison["cutoff"] == cutoff_date.date().isoformat()
].reset_index(drop=True)

selected_comparison


In [ ]:
# Save the chronological split assignments.
chronological_split_path = splits_dir / "chronological_split.csv"
chronological_split.to_csv(chronological_split_path, index=False)

print("Saved to:", chronological_split_path)
print("File exists:", chronological_split_path.exists())
print("File shape:", chronological_split.shape)


In [ ]:
# Reload and validate the exported chronological split.
chronological_check = pd.read_csv(
    chronological_split_path,
    parse_dates=["scheduled_arrival_date"]
)

assert chronological_check.equals(chronological_split.reset_index(drop=True))
assert chronological_check["row_id"].is_unique
assert set(chronological_check["row_id"]) == set(df["row_id"])
assert chronological_check["scheduled_arrival_date"].notna().all()

print("Loaded shape:", chronological_check.shape)
print("Split counts:")
print(chronological_check["split"].value_counts())
print("Export validation passed.")


In [ ]:
# Load random split assignments
random_split_check = pd.read_csv(
    splits_dir / "random_split.csv"
)

# Check both split files
for name, split_df in [
    ("Random", random_split_check),
    ("Chronological", chronological_check)
]:
    print(f"\n{name} split:")

    print("Shape:", split_df.shape)

    print(
        "Unique row IDs:",
        split_df["row_id"].is_unique
    )

    print(
        "All rows included:",
        set(split_df["row_id"]) == set(df["row_id"])
    )

    print(
        "Valid split labels:",
        set(split_df["split"]) == {"train", "test"}
    )

## Duplicate-overlap diagnostic

The anonymised dataset has no true booking identifier, so identical feature rows are not automatically deleted. The shared random split remains the common comparison split for all team members. However, exact feature overlap across training and test data can make random-holdout performance optimistic. The final robustness analysis must therefore also report the chronological test and a grouped or de-duplicated sensitivity check.


In [ ]:
# Duplicate-overlap diagnostic for the shared random split
official_classification_features = [
    "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled",
    "total_guests",
    "total_nights", "family_booking", "hotel",
    "arrival_date_month", "meal", "country",
    "market_segment", "distribution_channel",
    "reserved_room_type", "agent", "lead_time_group"
]

assert len(official_classification_features) == 24
assert set(official_classification_features).issubset(df.columns)

random_train = df[df["row_id"].isin(train_check["row_id"])]
random_test = df[df["row_id"].isin(test_check["row_id"])]

train_fingerprints = set(
    pd.util.hash_pandas_object(
        random_train[official_classification_features], index=False
    ).astype("uint64")
)
test_fingerprints = pd.util.hash_pandas_object(
    random_test[official_classification_features], index=False
).astype("uint64")
test_overlap = test_fingerprints.isin(train_fingerprints)

print("Test rows with an identical 24-feature row in training:",
      int(test_overlap.sum()))
print("Overlap rate (%):", round(test_overlap.mean() * 100, 2))
print("Shared split IDs remain unchanged: True")
print("Required robustness checks: chronological and grouped/de-duplicated sensitivity tests")
